# Import

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import math
import seaborn as sns
import pickle
import copy
from pathlib import Path



import torch
from torch import nn, Tensor

import time
import joblib

from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
from flow_matching.solver import Solver, ODESolver
from flow_matching.utils import ModelWrapper

from sklearn.preprocessing import StandardScaler

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module='torch')

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from fm.seed import set_seed
import fm.data as data
import fm.model as model
import fm.training as training
import fm.physics as physics
import fm.massesPlots as massesPlots
import fm.lossPlots as lossPlots
import fm.analysis as analysis
import fm.scaling_utils as scalingUtils

In [3]:
# for the plots :
import mplhep as hep

plt.style.use(hep.style.CMS)

plt.rcParams.update({
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 13,

    "lines.linewidth": 2,

    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "-",

    "legend.frameon": False,
})


# Utils

In [4]:
DATASET_TAG = "unbalanced"   # o balanced
PREPARED_DIR = "../results/prepared_splits"

SCALING_MODES = [
    "X_NONE__Y_NONE",
    "X_GLOBAL__Y_GLOBAL",
    "X_GLOBAL__Y_LOCAL",
    "X_LOCAL__Y_LOCAL",
    "X_NONE__Y_LOCAL",
]

CLASS_COL = "class"

In [5]:
tau1_cols = [
    "tau1_logpt", "tau1_eta", "tau1_phi", "tau1_mass",
    "tau1_dxy", "tau1_dz",
    "tau1_ptCorrPNet", "tau1_rawPNetVSjet", "tau1_rawDeepTau2018v2p5VSjet",
    "tau1_charge",
    "tau1_dM_0", "tau1_dM_1", "tau1_dM_2", "tau1_dM_10", "tau1_dM_11",
    "tau1_leadTkDeltaEta", "tau1_leadTkDeltaPhi", "tau1_leadTkPtOverTauPt",
]

tau2_cols = [
    "tau2_logpt", "tau2_eta", "tau2_phi", "tau2_mass",
    "tau2_dxy", "tau2_dz",
    "tau2_ptCorrPNet", "tau2_rawPNetVSjet", "tau2_rawDeepTau2018v2p5VSjet",
    "tau2_charge",
    "tau2_dM_0", "tau2_dM_1", "tau2_dM_2", "tau2_dM_10", "tau2_dM_11",
    "tau2_leadTkDeltaEta", "tau2_leadTkDeltaPhi", "tau2_leadTkPtOverTauPt", 
]

jet_cols = [
    "jet1_logpt", "jet1_eta", "jet1_phi", "jet1_mass",
    "jet2_logpt", "jet2_eta", "jet2_phi", "jet2_mass",
    "jet3_logpt", "jet3_eta", "jet3_phi", "jet3_mass",
]

met_cols = [
    "MET_logpt","MET_phi","MET_significance","MET_sumEt"
]

train_cols = (
    tau1_cols
    + tau2_cols
    + jet_cols
    + met_cols
)

y_cols = ["tau1_corr", "tau2_corr"]

# Loading & Preparing the DataFrames

In [6]:
with open("df_H_event.pkl", "rb") as f:
    df_H_event = pickle.load(f)

with open("df_DY_event.pkl", "rb") as f:
    df_DY_event = pickle.load(f)

In [7]:
# Creo i dataframe : 
#   1. All
#   2. Bilanciato

df_all = (
    pd.concat([df_DY_event, df_H_event], axis=0)
      .sample(frac=1.0, random_state=0)
      .reset_index(drop=True)
)

df_bal = data.make_balanced_df(df_all)

print("df_all:", df_all.shape)
print("df_bal:", df_bal.shape)
print("Counts df_all:", df_all["class"].value_counts().to_dict())
print("Counts df_bal:", df_bal["class"].value_counts().to_dict())

df_all: (337351, 76)
df_bal: (163894, 76)
Counts df_all: {1: 255404, 0: 81947}
Counts df_bal: {1: 81947, 0: 81947}


In [8]:
# -----------------------
# Split raw data
# -----------------------

df_all_train, df_all_val, df_all_test = data.split_df_once(df_all)
df_bal_train, df_bal_val, df_bal_test = data.split_df_once(df_bal)

print("ALL splits")
print("train:", df_all_train.shape)
print("val  :", df_all_val.shape)
print("test :", df_all_test.shape)

print()

print("BALANCED splits")
print("train:", df_bal_train.shape)
print("val  :", df_bal_val.shape)
print("test :", df_bal_test.shape)

ALL splits
train: (269880, 76)
val  : (33735, 76)
test : (33736, 76)

BALANCED splits
train: (131115, 76)
val  : (16389, 76)
test : (16390, 76)


In [9]:
# Scelgo un mode e prendo i risultati unbalanced 
BASELINE_MODE = "X_LOCAL__Y_LOCAL"   

m_dy_npz = np.load("../results/masses/masses_unbal_nonflat_mse_dy.npz")
m_h_npz  = np.load("../results/masses/masses_unbal_nonflat_mse_h.npz")

m_baseline_dy = m_dy_npz[BASELINE_MODE]
m_baseline_h  = m_h_npz[BASELINE_MODE]

# check:
print("DY baseline:", m_baseline_dy.shape)
print("H baseline:", m_baseline_h.shape)

DY baseline: (8244,)
H baseline: (25492,)


# Cumulative Trainings

In [10]:
ordered_features = [
    "jet1_logpt",
    "tau1_logpt",
    "tau2_logpt",
    "MET_sumEt",
    "tau1_dM_0",
    "jet2_logpt",
    "tau2_phi",
    "tau2_eta",
    "tau1_mass",
    "jet3_phi",
    "tau1_dM_2",
    "jet1_mass",
    "tau2_leadTkPtOverTauPt",
    "tau2_rawDeepTau2018v2p5VSjet",
    "tau2_leadTkDeltaEta",
    "jet2_phi",
    "jet3_logpt",
    "jet2_mass",
    "tau2_dM_11",
    "tau2_dM_1",
    "tau2_leadTkDeltaPhi",
    "tau2_dM_10",
    "tau1_rawDeepTau2018v2p5VSjet",
    "tau2_ptCorrPNet",
    "tau1_charge",
    "tau2_charge",
    "MET_phi",
    "tau2_dM_0",
    "tau2_mass",
    "tau1_eta",
    "tau1_dM_11",
    "tau1_rawPNetVSjet",
    "jet1_phi",
    "tau2_dz",
    "jet3_mass",
    "jet2_eta",
    "tau1_dM_1",
    "tau1_dM_10",
    "jet1_eta",
    "tau1_dz",
    "tau2_dM_2",
    "tau2_dxy",
    "tau1_phi",
    "MET_significance",
    "tau1_dxy",
    "tau1_leadTkDeltaPhi",
    "MET_logpt",
    "tau1_leadTkPtOverTauPt",
    "tau2_rawPNetVSjet",
    "tau1_leadTkDeltaEta",
    "jet3_eta",
    "tau1_ptCorrPNet"
]

In [13]:
save_dir = Path("../results/cumulative_scan")
save_dir.mkdir(exist_ok=True)

for k in range(1, len(ordered_features) + 1):
    outfile = save_dir / f"top{k}.pkl"

    if outfile.exists():
        print(f"[skip] top{k} already done")
        continue

    selected = ordered_features[:k]

    print("\n" + "="*80)
    print(f"TRAINING CUMULATIVE FEATURES ({k}/{len(ordered_features)}): {selected}")
    print("="*80)

    res_local = scalingUtils.train_selected_features_variant(
        df_train=df_all_train,
        df_val=df_all_val,
        df_test=df_all_test,
        selected_features=selected,
        variant="local",
        train_cols=train_cols,
        y_cols=y_cols,
    )

    result = {
        "features": selected,
        "local": res_local,
    }

    with open(outfile, "wb") as f:
        pickle.dump(result, f)


TRAINING CUMULATIVE FEATURES (1/52): ['jet1_logpt']
[1feat | local | ep    1] train=1.791469  val=1.111147  best=1.111147  bad=0/30
[1feat | local | ep   10] train=1.012991  val=0.989867  best=0.989867  bad=0/30
[1feat | local | ep   20] train=1.305242  val=0.867005  best=0.794544  bad=1/30
[1feat | local | ep   30] train=0.704944  val=0.669739  best=0.669739  bad=0/30
[1feat | local | ep   40] train=0.676760  val=0.656357  best=0.640770  bad=9/30
[1feat | local | ep   50] train=0.636579  val=0.587357  best=0.587357  bad=0/30
[1feat | local | ep   60] train=0.600488  val=0.667567  best=0.572086  bad=8/30
[1feat | local | ep   70] train=0.573826  val=0.596407  best=0.537940  bad=3/30
[1feat | local | ep   80] train=0.527324  val=0.523462  best=0.514028  bad=3/30
[1feat | local | ep   90] train=0.549721  val=0.584028  best=0.497768  bad=1/30
[1feat | local | ep  100] train=0.912026  val=0.901250  best=0.497768  bad=11/30
[1feat | local | ep  110] train=0.667664  val=0.650979  best=0.497

OSError: [Errno 28] No space left on device

In [ ]:
#predizioni: 
preds_cumulative_scan_all = {}

for key, res_pair in results_cumulative_scan_all.items():

    print("\n" + "="*80)
    print(f"PREDICTIONS CUMULATIVE: {key} -> {len(res_pair['features'])} features")
    print("="*80)

    res_local = res_pair["local"]
    df_test_local = res_local["df_splits_scaled"][2]

    res_local_pred = dict(res_local)
    res_local_pred["scalers"] = dict(res_local["scalers"])
    res_local_pred["scalers"]["mode"] = "X_NONE__Y_NONE"

    df_pred_local = model.predict_tau_corr_auto(
        df=df_test_local,
        res=res_local_pred,
        train_cols=train_cols,
        y_cols=y_cols,
    )

    preds_cumulative_scan_all[key] = {
        "features": res_pair["features"],
        "local": df_pred_local,
    }

In [ ]:
# masse
masses_cumulative_scan_all = {}
masses_cumulative_scan_all_dy = {}
masses_cumulative_scan_all_h = {}

for key, pred_pair in preds_cumulative_scan_all.items():

    print("\n" + "="*80)
    print(f"MASSES CUMULATIVE: {key} -> {len(pred_pair['features'])} features")
    print("="*80)

    df_pred_local = pred_pair["local"]

    m_local = physics.inv_mass_two_taus_corrected_ratio(
        df_all_test["tau1_pt_reco_corrPNet"].to_numpy(),
        df_all_test["tau1_eta"].to_numpy(),
        df_all_test["tau1_phi"].to_numpy(),
        df_all_test["tau1_mass"].to_numpy(),
        df_pred_local["tau1_corr_pred"].to_numpy(),
        df_all_test["tau2_pt_reco_corrPNet"].to_numpy(),
        df_all_test["tau2_eta"].to_numpy(),
        df_all_test["tau2_phi"].to_numpy(),
        df_all_test["tau2_mass"].to_numpy(),
        df_pred_local["tau2_corr_pred"].to_numpy(),
    )

    masses_cumulative_scan_all[key] = {
        "features": pred_pair["features"],
        "local": m_local,
    }

    masses_cumulative_scan_all_dy[key] = {
        "features": pred_pair["features"],
        "local": m_local[mask_dy],
    }

    masses_cumulative_scan_all_h[key] = {
        "features": pred_pair["features"],
        "local": m_local[mask_h],
    }

In [ ]:
rows = []

for key in masses_cumulative_scan_all.keys():

    res_sb = analysis.compute_SB_from_split_masses(
        masses_h=masses_cumulative_scan_all_h[key]["local"],
        masses_dy=masses_cumulative_scan_all_dy[key]["local"],
        window=WINDOW,
    )

    rows.append({
        "step": key,
        "n_features": len(masses_cumulative_scan_all[key]["features"]),
        "features": ", ".join(masses_cumulative_scan_all[key]["features"]),
        "N_S": res_sb["N_S"],
        "N_B": res_sb["N_B"],
        "S_over_B": res_sb["S_over_B"],
        "S_over_sqrtB": res_sb["S_over_sqrtB"],
    })

df_sb_cumulative_all = pd.DataFrame(rows).sort_values("n_features").reset_index(drop=True)
df_sb_cumulative_all

In [ ]:
baseline_rows = []

for mode in ["X_LOCAL__Y_LOCAL", "X_GLOBAL__Y_GLOBAL"]:
    res_sb = analysis.compute_SB_from_split_masses(
        masses_h=m_h_npz[mode],
        masses_dy=m_dy_npz[mode],
        window=WINDOW,
    )

    baseline_rows.append({
        "step": f"BASELINE_{mode}",
        "n_features": np.nan,
        "features": mode,
        "N_S": res_sb["N_S"],
        "N_B": res_sb["N_B"],
        "S_over_B": res_sb["S_over_B"],
        "S_over_sqrtB": res_sb["S_over_sqrtB"],
    })

df_sb_cumulative_all_full = pd.concat(
    [pd.DataFrame(baseline_rows), df_sb_cumulative_all],
    ignore_index=True
)

df_sb_cumulative_all_full